In [2]:
import pandas as pd

In [12]:
df_order = pd.read_csv('order.csv', sep=';')
df_payment = pd.read_csv('pay.csv', sep=';')
df_user = pd.read_csv('user.csv', sep=';')

In [10]:
df_order.isnull().sum()

order_id           0
symbol_id         69
buyer_id           0
order_status       0
order_platform    85
gmt_create         0
gmt_modified       0
dtype: int64

In [13]:
df_payment.isnull().sum()

payment_id        0
order_id          0
pay_method      158
pay_status        0
gmt_create        0
gmt_modified      0
dtype: int64

In [14]:
df_user.isnull().sum()

user_id                    0
registration_site          0
registration_platform    219
is_premium_user            0
registration_time          0
dtype: int64

In [3]:
df_sample = pd.read_csv('sample.csv')

In [4]:
df_sample

,Day,Open,High,Low,Close,Volume
0,1,47.04,48.24,47.04,48.15,3509
1,2,48.17,48.89,47.97,48.36,4862
2,3,48.55,49.71,48.52,49.55,1810
3,4,49.55,49.87,48.51,49.41,3824
4,5,49.41,49.96,45.84,46.36,2209
5,6,46.35,46.51,44.61,45.60,4558
6,7,45.54,45.54,43.63,44.02,3832
7,8,44.05,44.49,43.19,43.57,3778
8,9,43.63,45.29,43.63,45.00,1005
9,10,44.96,44.96,42.88,43.44,4047


In [5]:
df_sample['Typical_Price'] = (df_sample['High'] + df_sample['Low'] + df_sample['Close']) / 3

In [6]:
df_sample

,Day,Open,High,Low,Close,Volume,Typical_Price
0,1,47.04,48.24,47.04,48.15,3509,47.810000
1,2,48.17,48.89,47.97,48.36,4862,48.406667
2,3,48.55,49.71,48.52,49.55,1810,49.260000
3,4,49.55,49.87,48.51,49.41,3824,49.263333
4,5,49.41,49.96,45.84,46.36,2209,47.386667
5,6,46.35,46.51,44.61,45.60,4558,45.573333
6,7,45.54,45.54,43.63,44.02,3832,44.396667
7,8,44.05,44.49,43.19,43.57,3778,43.750000
8,9,43.63,45.29,43.63,45.00,1005,44.640000
9,10,44.96,44.96,42.88,43.44,4047,43.760000


In [7]:
df_sample['Money_Flow'] = None
for i in range(1, len(df_sample)):
    money_flow = df_sample.loc[i, 'Typical_Price'] * df_sample.loc[i, 'Volume']
    if df_sample.loc[i, 'Typical_Price'] > df_sample.loc[i - 1, 'Typical_Price']:
        df_sample.loc[i, 'Money_Flow'] = money_flow
    else:
        df_sample.loc[i, 'Money_Flow'] = -money_flow

In [8]:
df_sample

,Day,Open,High,Low,Close,Volume,Typical_Price,Money_Flow
0,1,47.04,48.24,47.04,48.15,3509,47.810000,None
1,2,48.17,48.89,47.97,48.36,4862,48.406667,235353.213333
2,3,48.55,49.71,48.52,49.55,1810,49.260000,89160.6
3,4,49.55,49.87,48.51,49.41,3824,49.263333,188382.986667
4,5,49.41,49.96,45.84,46.36,2209,47.386667,-104677.146667
5,6,46.35,46.51,44.61,45.60,4558,45.573333,-207723.253333
6,7,45.54,45.54,43.63,44.02,3832,44.396667,-170128.026667
7,8,44.05,44.49,43.19,43.57,3778,43.750000,-165287.5
8,9,43.63,45.29,43.63,45.00,1005,44.640000,44863.2
9,10,44.96,44.96,42.88,43.44,4047,43.760000,-177096.72


In [11]:
def calculate_mfi(df_sample, m):
    # Validate m is within acceptable range
    if m < 2 or m > len(df_sample):
        raise ValueError(f"m must be between 2 and {len(df_sample)}, got {m}")
    # Copy df_sample to avoid modifying the original
    df_result = df_sample.copy()
    # Initialize new columns with None
    df_result['Positive_Money_Flow_Sum'] = None
    df_result['Negative_Money_Flow_Sum'] = None
    df_result['MFI'] = None
    for i in range(m, len(df_result)):
        # Get the last m rows of Money_Flow
        window = df_result['Money_Flow'].iloc[i - m:i]
        # Sum of positive money flows in the window
        positive_sum = window[window > 0].sum()
        # Sum of negative money flows (use absolute value)
        negative_sum = abs(window[window < 0].sum())
        # Store sums
        df_result.loc[df_result.index[i], 'Positive_Money_Flow_Sum'] = positive_sum
        df_result.loc[df_result.index[i], 'Negative_Money_Flow_Sum'] = negative_sum
        # Calculate MFI
        # Guard against division by zero: if negative_sum is 0,
        # MFI is set to 100 (all positive flows, maximum value)
        if negative_sum == 0:
            df_result.loc[df_result.index[i], 'MFI'] = 100.0
        else:
            money_ratio = positive_sum / negative_sum
            df_result.loc[df_result.index[i], 'MFI'] = 100 * money_ratio / (1 + money_ratio)
    return df_result

for m in [5, 7, 10]:
    df_output = calculate_mfi(df_sample, m)
    output_filename = f'sample_output_{m}.csv'
    df_output.to_csv(output_filename, index=False)
    print(f"Generated {output_filename}")
print("All done.")

Generated sample_output_5.csv
Generated sample_output_7.csv
Generated sample_output_10.csv
All done.
